In [ ]:
#pip install langchain-community langchain-core

In [ ]:
#!pip install langchain
#!pip install langchain-openai
#!pip install chromadb
#!pip install pypdf

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chains import RetrievalQAWithSourcesChain
from langchain_openai import OpenAI
import os
import shutil

In [ ]:
openai_api_key = 'SOME_KEY'

In [ ]:
categories = ['aerospace', 'automotive', 'biomedical']
base_path = 'datasets'
persist_directory = "chroma"

In [ ]:
# Function to create a vectore store
def create_or_load_vector_store(categories):

    vector_store_directory = os.path.join(persist_directory, "all_categories")
    
    # If the vector store already exists, then load it
    if os.path.exists(vector_store_directory): 
        print("Loading existing vector store for all categories")
        vectordb = Chroma(
            persist_directory=vector_store_directory,
            embedding_function=OpenAIEmbeddings(openai_api_key=openai_api_key)
        )
    else: # create new vector store
        print("Creating new vector store for all categories")
        all_documents = []

        # collect all documents from every field
        for category in categories:
            pdf_folder_path = os.path.join(base_path, f"{category}_new") # check for the folders
            documents = []

            for file in os.listdir(pdf_folder_path): # going through each PDF 
                if file.endswith('.pdf'):
                    try:
                        pdf_path = os.path.join(pdf_folder_path, file)
                        loader = PyPDFLoader(pdf_path)
                        documents.extend(loader.load())
                    except Exception as e:
                        print(f"Error loading {file}: {e}")

            all_documents.extend(documents) # new list to combine all documents to one list, which will then be converted to vectors

        print(f"Total number of documents across all categories: {len(all_documents)}")

        # Devision to chunks
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=10)
        chunked_documents = text_splitter.split_documents(all_documents)

        # Creating vector store from chunked documents
        vectordb = Chroma.from_documents(
            documents=chunked_documents,
            embedding=OpenAIEmbeddings(openai_api_key=openai_api_key),
            persist_directory=vector_store_directory
        )
        vectordb.persist()

    print("Creating or Loading the database is done.")

    return vectordb

    # Load docs -> split text into smaller chunks -> convert each chunk into embeddings (vectors (a list of numbers)) that represent semantic meaning ->
    # -> Store generated vectors into Chroma db 

In [ ]:
# New function to add articles to the existing vector store
def add_articles_to_vector_store(additional_category):
   
    vector_store_directory = os.path.join(persist_directory, "all_categories")
    
    if os.path.exists(vector_store_directory): # if the vector store already exists
        print("Loading existing vector store for all categories")
        vectordb = Chroma(
            persist_directory=vector_store_directory,
            embedding_function=OpenAIEmbeddings(openai_api_key=openai_api_key)
        )
    else: 
        raise Exception("Vector store does not exist. Create it first using create_or_load_vector_store().")

    # Collect documents from the additional category
    pdf_folder_path = os.path.join(base_path, f"{additional_category}")
    documents = []

    for file in os.listdir(pdf_folder_path):
        if file.endswith('.pdf'):
            try:
                pdf_path = os.path.join(pdf_folder_path, file)
                loader = PyPDFLoader(pdf_path)
                documents.extend(loader.load())
            except Exception as e:
                print(f"Error loading {file}: {e}")

    print(f"Total number of additional documents: {len(documents)}")

    # Division into chunks
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=10)
    chunked_documents = text_splitter.split_documents(documents)

    # Adding new documents to vector store
    vectordb.add_documents(chunked_documents)
    vectordb.persist()

    return vectordb

In [ ]:
vector_stores = create_or_load_vector_store(categories) # create ChromaDB

In [ ]:
vector_stores = add_articles_to_vector_store('additional_new') # Add new articles

In [ ]:
# Defining the model
llm = OpenAI(model_name="gpt-3.5-turbo-instruct", temperature=0, openai_api_key=openai_api_key)

In [ ]:
# function that will generate responses
def generate_response(query, vector_stores, top_k=6): 

    retriever = vector_stores.as_retriever(top_k=top_k)
    chain = RetrievalQAWithSourcesChain.from_chain_type(
        llm=llm, chain_type="stuff", retriever=retriever)
    result = chain({"question": query})

    sources = result['sources'].split(', ')  # Assuming sources are separated by commas
    formatted_sources = "\n".join([f"{i+1}) {source}" for i, source in enumerate(sources)])
    response = f"Answer: {result['answer']}\nSources:\n{formatted_sources}\n"

    return response


In [ ]:
# Asking questions
query = "What are the mechanical properties of the metals SS316L, Ti6Al4V, and AlSi10Mg I want to use in additive manufacturing? Give properties for each metal."
response = generate_response(query, vector_stores, top_k=6)
print(response)